# Towards Using Data Mining and Machine Learning for Diabetes Risk Prediction

## Import the data and quick inspection

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/cepulmano/msys116-datasets/refs/heads/main/diabetes_data.csv")
df.head()

## RQ: How can we accurately predict diabetes risk using machine learning?

## Exploratory Data Analysis

In [ ]:
# Descriptive statistics
df.describe()

In [ ]:
# Explore the distribution of the target variable
df['Diabetes_binary'].value_counts()

In [ ]:
# Visualize the distribution of the target variable
import matplotlib.pyplot as plt
import seaborn as sns

sns.countplot(x='Diabetes_binary', data=df)
plt.show()

In [ ]:
# Explore the relationship between different features and the target variable
# Example: Boxplot of BMI for diabetic and non-diabetic individuals
sns.boxplot(x='Diabetes_binary', y='BMI', data=df)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='BMI', hue='Diabetes_binary', kde=True)
plt.title('Distribution of BMI by Diabetes Status')
plt.show()


In [ ]:
# Calculate the correlation matrix
correlation_matrix = df.corr()

# Visualize the correlation matrix using a heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.show()

## Classification Using Multilayer Perceptrons Neural Networks on Imbalanced Dataset

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Separate features (X) and target variable (y)
X = df.drop('Diabetes_binary', axis=1)
y = df['Diabetes_binary']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the features using StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Initialize and train the MLPClassifier
mlp = MLPClassifier(hidden_layer_sizes=(10), max_iter=1000)
mlp.fit(X_train, y_train)

# Make predictions on the test set
predictions = mlp.predict(X_test)

# Evaluate the model
print(classification_report(y_test, predictions))
print(confusion_matrix(y_test,predictions))
# About metrics: https://www.labelf.ai/blog/what-is-accuracy-precision-recall-and-f1-score

tn, fp, fn, tp = confusion_matrix(y_test,predictions).ravel().tolist()
print(f"\nTrue Negative: {tn}")
print(f"False Positive: {fp}")
print(f"False Negative: {fn}")
print(f"True Positive: {tp}")

#### **Recall (True Positive Rate)** - out of all the patients with Diabetes, how many did we predict accurately? TP / (TP + FN)
#### **Precision (Positive Predictive Value)** - if the model predicts a patient as positive for Diabetes, what is the probability that the patient actually has Diabetes? TP / (TP + FP)
#### **False Positive Rate** - out of all the patients without Diabetes, how many did the model predict positively?

## Preprocessing

### Balance the Dataset Using Undersampling Method

In [ ]:
from imblearn.under_sampling import RandomUnderSampler

# Separate features (X) and target variable (y)
X = df.drop('Diabetes_binary', axis=1)
y = df['Diabetes_binary']

# Initialize RandomUnderSampler
undersampler = RandomUnderSampler(random_state=42)

# Resample the dataset
X_resampled, y_resampled = undersampler.fit_resample(X, y)

# Create a new balanced DataFrame
balanced_df = pd.DataFrame(X_resampled, columns=X.columns)
balanced_df['Diabetes_binary'] = y_resampled

# Display class counts in the balanced dataset
print(balanced_df['Diabetes_binary'].value_counts())


## Clustering Using KMeans Clustering

In [ ]:
from sklearn.cluster import KMeans

# Select features for clustering (excluding the target variable)
features = balanced_df.drop('Diabetes_binary', axis=1)

# Determine the optimal number of clusters (e.g., using the elbow method)
inertia = []
for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=0)
    kmeans.fit(features)
    inertia.append(kmeans.inertia_)

plt.plot(range(1, 11), inertia, marker='o')
plt.title('Elbow Method for Optimal k')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.show()

# Apply KMeans clustering with the chosen number of clusters (e.g., k=3)
k = 3  # Replace with the optimal k value from the elbow method
kmeans = KMeans(n_clusters=k, random_state=0)
balanced_df['cluster'] = kmeans.fit_predict(features)

In [ ]:
# Visualize the clusters (example with two selected features)
plt.figure(figsize=(8, 6))
sns.scatterplot(x='BMI', y='Age', hue='cluster', data=balanced_df, palette='viridis')
plt.scatter(kmeans.cluster_centers_[:, balanced_df.columns.get_loc('BMI')],
            kmeans.cluster_centers_[:, balanced_df.columns.get_loc('Age')],
            s=200, c='red', label='Centroids')  # Plot centroids
plt.title('KMeans Clustering')
plt.legend()
plt.show()

## Classification Using Multilayer Perceptrons Neural Networks on Balanced Dataset with 5-fold Cross Validation

In [ ]:
from sklearn.model_selection import KFold

# Separate features (X) and target variable (y)
X = balanced_df.drop('Diabetes_binary', axis=1)
y = balanced_df['Diabetes_binary']

# Define the number of folds for cross-validation
n_splits = 5

# Initialize KFold
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Initialize lists to store evaluation metrics for each fold
accuracy_scores = []
precision_scores = []
recall_scores = []
f1_scores = []

# Perform 5-fold cross-validation
for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Scale the features
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Initialize and train the MLPClassifier
    mlp = MLPClassifier(hidden_layer_sizes=(10), max_iter=1000)
    mlp.fit(X_train, y_train)

    # Make predictions
    predictions = mlp.predict(X_test)

    # Evaluate the model and store the metrics
    report = classification_report(y_test, predictions, output_dict=True)
    accuracy_scores.append(report['accuracy'])
    precision_scores.append(report['macro avg']['precision'])
    recall_scores.append(report['macro avg']['recall'])
    f1_scores.append(report['macro avg']['f1-score'])

# Print the average performance metrics across all folds
# About metrics: https://www.labelf.ai/blog/what-is-accuracy-precision-recall-and-f1-score
print(f"Average Accuracy: {np.mean(accuracy_scores)}")
print(f"Average Precision: {np.mean(precision_scores)}")
print(f"Average Recall: {np.mean(recall_scores)}")
print(f"Average F1-score: {np.mean(f1_scores)}")


## Build Model for Diabetes Risk Prediction System

In [ ]:
# Separate features (X) and target variable (y)
X = balanced_df.drop('Diabetes_binary', axis=1)
y = balanced_df['Diabetes_binary']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the features using StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Initialize and train the MLPClassifier
mlp = MLPClassifier(hidden_layer_sizes=(10), max_iter=1000)
mlp.fit(X_train, y_train)

# Make predictions on the test set
predictions = mlp.predict(X_test)

# Evaluate the model
print(classification_report(y_test, predictions))
print(confusion_matrix(y_test, predictions))

tn, fp, fn, tp = confusion_matrix(y_test,predictions).ravel().tolist()
print(f"\nTrue Negative: {tn}")
print(f"False Positive: {fp}")
print(f"False Negative: {fn}")
print(f"True Positive: {tp}")

In [ ]:
from sklearn.metrics import roc_auc_score

# Get probability predictions for the positive class (class 1)
probabilities = mlp.predict_proba(X_test)[:, 1]

# Calculate the AUROC score
auroc = roc_auc_score(y_test, probabilities)

print(f"AUROC Score: {auroc}")

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# Calculate the false positive rate, true positive rate, and thresholds
fpr, tpr, thresholds = roc_curve(y_test, probabilities)

# Calculate the Area Under the Curve (AUC)
roc_auc = auc(fpr, tpr)

# Plot the ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

## Predict Diabetes Risk using New Data

In [ ]:
# New input data
new_data = pd.DataFrame({
    'HighBP': [1], 'HighChol': [0], 'CholCheck': [1], 'BMI': [25], 'Smoker': [0],
    'Stroke': [0], 'HeartDiseaseorAttack': [0], 'PhysActivity': [1], 'Fruits': [1],
    'Veggies': [1], 'HvyAlcoholConsump': [0], 'AnyHealthcare': [1], 'NoDocbcCost': [0],
    'GenHlth': [3], 'MentHlth': [1], 'PhysHlth': [0], 'DiffWalk': [0], 'Sex': [1],
    'Age': [45], 'Education': [6], 'Income': [8], 'cluster': [1]
})

# Scale the new data using the same scaler used for training
new_data_scaled = scaler.transform(new_data)

# Make predictions on the new data
new_prediction = mlp.predict(new_data_scaled)
print(f"Prediction for new input: {new_prediction[0]}")